# nb4c — Phase 2 (Bước 1): Audit người 25 mẫu adjudication (blind) + đối chiếu nhãn LLM

Notebook **CPU, không train, không fit tham số nào** — chỉ đo lường đồng thuận giữa người soát và
nhãn LLM (nb4b, model `jev-latest`) trên bộ 100 mẫu adjudication của nb4. Chạy được **cục bộ**
(`./data`) hoặc trên Kaggle (attach Input) — cùng convention tự dò input với nb4b.

**Bối cảnh**: lần chạy nb4 24/09 cho nhãn **99A/1C** → nhiễu pseudo-gold (B+D) = **0,0%**
(CI95 [0; 3,7%], n=100). Một annotator LLM duy nhất + gần như đồng thuận tuyệt đối trên nhóm
*suspect* là kết quả "sạch bất thường" → bắt buộc audit người trước khi chốt con số vào REPORT.

**Thiết kế audit (pre-registered, 25 mẫu blind)**:
- Thành phần: 2 mẫu `llm_confidence` thấp nhất + toàn bộ stratum `>=0.5` + toàn bộ nhãn `C` +
  random seed-42 từ nhãn `A` điền đủ 25 (dedupe theo `sample_id`; ưu tiên low-conf → ≥0.5 → C → random).
- **Blind tuyệt đối**: `audit_samples.json` không chứa nhãn LLM hay confidence (có assert).
- Quy tắc quyết định — chốt **trước** khi xem kết quả (dựa trên đồng thuận **binary**: OK=`A/C` vs NHIỄU=`B/D`):
  - người khớp LLM **≥ 24/25** → chốt nhiễu 0% (human-audited), ghi consensus vào REPORT;
  - **20–23/25** → mở rộng audit toàn bộ 100 mẫu (cell `export_round2`) trước khi kết luận;
  - người phát hiện mẫu lệch nhãn **B/D ↔ A/C** → tạo `adjudication_filled_v2.json` (nhãn người
    là chuẩn) + **tính lại nhiễu + Wilson CI trên 100 nhãn hiệu chỉnh**.
- **Fallback** (chưa có `adjudication_filled.json` local): chọn từ `adjudication_samples.json`
  (template nb4) theo phương án suy biến **23 random + toàn bộ stratum `>=0.5`**; khi đó không
  tính được consensus-vs-LLM → cell đối chiếu trả về **ước lượng nhiễu theo nhãn người** + Wilson CI
  (estimator thay thế, ghi rõ trong REPORT).

**Quy trình**: chạy cell "Chọn mẫu" → mở `data/audit_samples.json` điền `label` ∈ {A,B,C,D}
(+ `label_note` nếu cần) → lưu thành `data/audit_filled.json` (hoặc điền thẳng trong file gốc)
→ chạy cell "Đối chiếu & consensus" → cell "REPORT-ready" in khối markdown dán vào REPORT.md.


In [ ]:
import json
import math
import random
import datetime
import collections
from pathlib import Path

NOTEBOOK = 'nb4c_adjudication_audit'
RUN_STAMP = datetime.datetime.now().isoformat(timespec='seconds')

# Buckets cấu hình — mọi hằng số gom một chỗ (DESIGN.md §7)
SEED = 42
AUDIT_N = 25
LOW_CONF_N = 2
CONSENSUS_ACCEPT = 24      # binary đồng thuận >= 24/25 → chốt
CONSENSUS_EXPAND_MIN = 20  # 20-23 → mở rộng audit 100 mẫu
VALID_LABELS = ('A', 'B', 'C', 'D')
BINARY_OK = ('A', 'C')     # không nhiễu
BINARY_NOISY = ('B', 'D')  # nhiễu pseudo-gold
Z = 1.959963984540054      # khớp công thức Wilson của nb4 §6

RUBRIC = {
    'A': 'Hai vế song song, các edit là lỗi chính tả thật → pseudo-gold OK',
    'B': 'Phía source đúng / bản sửa sai → pseudo-gold NHIỄU (model bị phạt oan)',
    'C': 'Edit phi chính tả (số liệu, viết tắt, tên riêng, format) — không tính là nhiễu',
    'D': 'Hai vế không song song / align hỏng → tính là nhiễu',
}

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('./data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def find_input(name, required=True):
    """Tự dò Input: /kaggle/input (Kaggle) trước, fallback ./data (chạy cục bộ)."""
    for root in (Path('/kaggle/input'), Path('./data')):
        if root.is_dir():
            hits = sorted(root.rglob(name))
            if hits:
                return hits[0]
    if required:
        raise FileNotFoundError('Không tìm thấy ' + name + ' trong /kaggle/input hoặc ./data')
    return None


def wilson_ci(p_hat, n, z=Z):
    """Wilson score interval — cùng công thức nb4 §6."""
    if n == 0:
        return None
    denom = 1 + z * z / n
    center = (p_hat + z * z / (2 * n)) / denom
    half = z * math.sqrt(p_hat * (1 - p_hat) / n + z * z / (4 * n * n)) / denom
    return [max(0.0, center - half), min(1.0, center + half)]


def classify_consensus(binary_agree, n_total):
    """Quy tắc pre-registered: accept / expand / reject (nhiễu thật)."""
    if binary_agree >= CONSENSUS_ACCEPT:
        return 'accept'
    if binary_agree >= CONSENSUS_EXPAND_MIN:
        return 'expand'
    return 'reject'


def sample_view(s):
    """Trường hiển thị cho người soát — KHÔNG bao giờ chứa nhãn LLM/confidence (blind)."""
    return {
        'sample_id': s['sample_id'], 'stratum': s['stratum'], 'test_index': s['test_index'],
        'text': s['text'], 'corrected_text': s['corrected_text'],
        'src_marked': s['src_marked'], 'tgt_marked': s['tgt_marked'],
        'edit_blocks': s['edit_blocks'], 'edit_ratio': s.get('edit_ratio'),
        'n_src_tokens': s.get('n_src_tokens'),
        'label': None, 'label_note': '',
    }


# Sanity — PASS bắt buộc trước khi chạy dữ liệu thật
_ci0 = wilson_ci(0.0, 100)
assert abs(_ci0[0]) < 1e-12 and abs(_ci0[1] - 0.0369960) < 1e-4, _ci0
assert classify_consensus(25, 25) == 'accept' and classify_consensus(24, 25) == 'accept'
assert classify_consensus(23, 25) == 'expand' and classify_consensus(20, 25) == 'expand'
assert classify_consensus(19, 25) == 'reject'
print('Sanity PASS: Wilson 0/100 → CI95 [0.0%%, %.1f%%] (khớp output nb4 24/09); quy tắc consensus OK'
      % (100 * _ci0[1]))


In [ ]:
# Cell "Chọn mẫu" — 25 mẫu blind → audit_samples.json
filled_path = find_input('adjudication_filled.json', required=False)
template_path = find_input('adjudication_samples.json')
if filled_path is not None:
    payload = json.loads(Path(filled_path).read_text(encoding='utf-8'))
    LLM_LABELS_AVAILABLE = True
    print('Nạp adjudication_filled.json:', filled_path)
else:
    assert template_path is not None, 'Thiếu cả adjudication_filled.json và adjudication_samples.json'
    payload = json.loads(Path(template_path).read_text(encoding='utf-8'))
    LLM_LABELS_AVAILABLE = False
    print('KHÔNG có adjudication_filled.json → FALLBACK từ adjudication_samples.json:')
    print('  thành phần suy biến: 23 random + toàn bộ stratum >=0.5 (không biết nhãn C/conf).')

samples = payload['samples']
assert len(samples) == payload.get('n_samples', len(samples)), 'payload.samples != n_samples'
if LLM_LABELS_AVAILABLE:
    assert all(s.get('label') in VALID_LABELS for s in samples), 'File filled còn mẫu thiếu nhãn'

rng = random.Random(SEED)


def _by_id(seq):
    return sorted(seq, key=lambda s: s['sample_id'])


audit_pick = []  # (reason, sample) — giữ thứ tự ưu tiên
if LLM_LABELS_AVAILABLE:
    for s in _by_id(sorted((x for x in samples if (x.get('llm_confidence') or 0.0) > 0),
                           key=lambda x: ((x.get('llm_confidence') or 0.0), x['sample_id']))[:LOW_CONF_N]):
        audit_pick.append(('low_conf', s))
    for s in _by_id(x for x in samples if x['stratum'] == '>=0.5'):
        audit_pick.append(('stratum_ge_0.5', s))
    for s in _by_id(x for x in samples if x['label'] == 'C'):
        audit_pick.append(('label_C', s))
else:
    for s in _by_id(x for x in samples if x['stratum'] == '>=0.5'):
        audit_pick.append(('stratum_ge_0.5', s))

seen = set()
dedup = []
for reason, s in audit_pick:
    if s['sample_id'] not in seen:
        seen.add(s['sample_id'])
        dedup.append((reason, s))
audit_pick = dedup

if LLM_LABELS_AVAILABLE:
    pool_a = _by_id(x for x in samples if x['label'] == 'A' and x['sample_id'] not in seen)
    reason = 'random_A'
else:
    pool_a = sorted((x for x in samples if x['sample_id'] not in seen), key=lambda x: x['test_index'])
    reason = 'random'
for s in rng.sample(pool_a, min(AUDIT_N - len(audit_pick), len(pool_a))):
    audit_pick.append((reason, s))

assert len(audit_pick) == AUDIT_N, 'Chỉ chọn được %d/%d mẫu' % (len(audit_pick), AUDIT_N)

audit_payload = {
    'created': RUN_STAMP, 'notebook': NOTEBOOK, 'seed': SEED,
    'n_audit': len(audit_pick),
    'llm_labels_available': LLM_LABELS_AVAILABLE,
    'source_file': str(filled_path if LLM_LABELS_AVAILABLE else template_path),
    'composition': dict(collections.Counter(r for r, _s in audit_pick)),
    'rubric': RUBRIC,
    'instructions': [
        'Soát MÙ: file này cố tình không chứa nhãn LLM/confidence — đừng tìm xem LLM ghi gì.',
        'Điền label in {A,B,C,D} theo rubric (A/C = pseudo-gold OK, B/D = NHIỄU) + label_note nếu cần.',
        'Lưu thành data/audit_filled.json (giữ nguyên schema) hoặc điền thẳng trong file này.',
        'Chạy lại cell "Đối chiếu & consensus" của nb4c.',
    ],
    'samples': [sample_view(s) for _r, s in audit_pick],
}
_blob = json.dumps(audit_payload, ensure_ascii=False)
assert 'llm_confidence' not in _blob and 'llm_probabilities' not in _blob and 'llm_model' not in _blob,     'BLIND LEAK: file audit đang chứa trường nhãn LLM'
assert all(s['label'] is None for s in audit_payload['samples'])

out_path = OUTPUT_DIR / 'audit_samples.json'
out_path.write_text(json.dumps(audit_payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('Đã xuất %s: %d mẫu blind (llm_labels_available=%s)' % (out_path, len(audit_pick), LLM_LABELS_AVAILABLE))
print('Thành phần:', audit_payload['composition'])
for s in audit_payload['samples'][:2]:
    print('—— %s · stratum=%s · edit_ratio=%s' % (s['sample_id'], s['stratum'], s['edit_ratio']))
    print('  SRC:', s['src_marked'])
    print('  TGT:', s['tgt_marked'])


In [ ]:
# Cell "Đối chiếu & consensus" — chạy SAU khi điền nhãn người
AUDIT = None
audit_file = find_input('audit_filled.json', required=False) or find_input('audit_samples.json', required=False)
if audit_file is None:
    print('Chưa có audit_filled.json — điền nhãn 25 mẫu rồi chạy lại cell này.')
else:
    audit = json.loads(Path(audit_file).read_text(encoding='utf-8'))
    a_samples = audit['samples']
    missing = [s['sample_id'] for s in a_samples if s.get('label') not in VALID_LABELS]
    if missing:
        print('Còn %d/%d mẫu chưa điền label (vd %s…).' % (len(missing), len(a_samples), missing[0]))
        print('→ Điền label in {A,B,C,D} rồi chạy lại cell này (rubric ở đầu file).')
    else:
        llm_map = {}
        if audit.get('llm_labels_available'):
            _src = json.loads(Path(find_input('adjudication_filled.json')).read_text(encoding='utf-8'))
            llm_map = {s['sample_id']: s for s in _src['samples']}

        rows = []
        for s in a_samples:
            h, l = s['label'], llm_map.get(s['sample_id'], {}).get('label')
            rows.append({
                'sample_id': s['sample_id'], 'human': h, 'llm': l,
                'binary_agree': None if l is None else (h in BINARY_NOISY) == (l in BINARY_NOISY),
                'full_agree': None if l is None else h == l,
            })

        if llm_map:
            n = len(rows)
            bin_ok = sum(1 for r in rows if r['binary_agree'])
            full_ok = sum(1 for r in rows if r['full_agree'])
            confusion = collections.Counter((r['llm'], r['human']) for r in rows)
            corrections = [{'sample_id': r['sample_id'], 'llm': r['llm'], 'human': r['human']}
                           for r in rows if not r['binary_agree']]
            decision = classify_consensus(bin_ok, n)

            v2_info = None
            if corrections:
                _src_path = find_input('adjudication_filled.json')
                v2 = json.loads(Path(_src_path).read_text(encoding='utf-8'))
                hum = {r['sample_id']: r['human'] for r in corrections}
                n_changed = 0
                for s in v2['samples']:
                    if s['sample_id'] in hum and s.get('label') != hum[s['sample_id']]:
                        s['label'] = hum[s['sample_id']]
                        n_changed += 1
                cnt = collections.Counter(s['label'] for s in v2['samples'])
                noise2 = (cnt['B'] + cnt['D']) / sum(cnt.values())
                v2['meta_audit'] = {
                    'created': RUN_STAMP, 'notebook': NOTEBOOK, 'seed': SEED,
                    'n_audit': n, 'binary_consensus': '%d/%d' % (bin_ok, n),
                    'corrections': corrections,
                }
                v2_path = OUTPUT_DIR / 'adjudication_filled_v2.json'
                v2_path.write_text(json.dumps(v2, ensure_ascii=False, indent=2), encoding='utf-8')
                v2_info = {'path': str(v2_path), 'n_changed': n_changed,
                           'noise_after': noise2, 'wilson95_ci_after': wilson_ci(noise2, sum(cnt.values()))}

            AUDIT = {'mode': 'consensus_vs_llm', 'n': n, 'binary_agree': bin_ok,
                     'binary_consensus': bin_ok / n, 'full_agree': full_ok,
                     'confusion': {'%s→%s' % k: v for k, v in sorted(confusion.items())},
                     'corrections': corrections, 'decision': decision, 'v2': v2_info,
                     'rows': rows}
            print('=== CONSENSUS (nhãn người vs LLM, %d mẫu) ===' % n)
            print('Binary đồng thuận (OK=A/C vs NHIỄU=B/D): %d/%d = %.0f%%' % (bin_ok, n, 100 * bin_ok / n))
            print('Full-label agreement: %d/%d' % (full_ok, n))
            print('Nhầm lẫn (llm→human):', dict(AUDIT['confusion']))
            if corrections:
                print('Mẫu lệch nhãn binary:')
                for r in corrections:
                    print('  %s: llm=%s → human=%s' % (r['sample_id'], r['llm'], r['human']))
            if v2_info:
                print('Đã ghi %s (sửa %d nhãn) → nhiễu sau hiệu chỉnh: %.1f%% (CI95 [%.1f%%, %.1f%%])'
                      % (v2_info['path'], v2_info['n_changed'], 100 * v2_info['noise_after'],
                         100 * v2_info['wilson95_ci_after'][0], 100 * v2_info['wilson95_ci_after'][1]))
            print('QUYẾT ĐỊNH (pre-registered):', decision,
                  '→ chốt nhiễu 0% (human-audited)' if decision == 'accept'
                  else '→ mở rộng audit 100 mẫu (cell export_round2)' if decision == 'expand'
                  else '→ pseudo-gold có nhiễu: dùng nhãn v2, đọc kèm caveat')
        else:
            cnt = collections.Counter(r['human'] for r in rows)
            n = len(rows)
            noise_h = (cnt['B'] + cnt['D']) / n
            AUDIT = {'mode': 'human_only', 'n': n, 'label_counts': dict(cnt),
                     'noise_human': noise_h, 'wilson95_ci': wilson_ci(noise_h, n), 'rows': rows}
            print('=== ƯỚC LƯỢNG NHIỄU THEO NHÃN NGƯỜI (không có nhãn LLM local — fallback) ===')
            print('Nhãn người:', dict(cnt))
            print('Nhiễu (B+D)/%d = %.1f%% · CI95 [%.1f%%, %.1f%%]' % (n, 100 * noise_h,
                  100 * AUDIT['wilson95_ci'][0], 100 * AUDIT['wilson95_ci'][1]))
            print('(> 8% → caveat bắt buộc; ≤ 8% → bổ sung bằng chứng pseudo-gold sạch;')
            print(' sau khi có adjudication_filled.json local, chạy lại cell để có consensus-vs-LLM)')


In [ ]:
# Cell "REPORT-ready" — khối markdown dán vào REPORT.md mục 7.3 (chỉ khi AUDIT đã có kết quả)
if AUDIT is None:
    print('Chưa có kết quả audit — chạy cell "Đối chiếu & consensus" trước.')
else:
    print('```markdown')
    if AUDIT['mode'] == 'consensus_vs_llm':
        print('- **Audit người (nb4c, 25 mẫu blind, seed 42)**: binary consensus %d/25 (%.0f%%) · '
              'full-label agreement %d/25 · quyết định: **%s**.'
              % (AUDIT['binary_agree'], 100 * AUDIT['binary_consensus'], AUDIT['full_agree'],
                 {'accept': 'CHẤT nhận nhiễu 0% (human-audited)',
                  'expand': 'MỞ RỘNG audit 100 mẫu trước khi kết luận',
                  'reject': 'pseudo-gold CÓ NHIỄU — dùng adjudication_filled_v2'}[AUDIT['decision']]))
        if AUDIT['v2']:
            print('  Hiệu chỉnh %d nhãn → nhiễu pseudo-gold sau audit: %.1f%% (CI95 [%.1f%%, %.1f%%]).'
                  % (AUDIT['v2']['n_changed'], 100 * AUDIT['v2']['noise_after'],
                     100 * AUDIT['v2']['wilson95_ci_after'][0], 100 * AUDIT['v2']['wilson95_ci_after'][1]))
    else:
        print('- **Audit người (nb4c, 25 mẫu blind, seed 42 — fallback không có nhãn LLM local)**: '
              'nhiễu theo nhãn người (B+D) = %.1f%% (CI95 [%.1f%%, %.1f%%], n=%d)'
              % (100 * AUDIT['noise_human'], 100 * AUDIT['wilson95_ci'][0],
                 100 * AUDIT['wilson95_ci'][1], AUDIT['n']))
        print('  (chưa thay thế được consensus-vs-LLM; cập nhật khi có adjudication_filled.json local)')
    print('```')


In [ ]:
# Cell "export_round2" — chỉ chạy khi quyết định consensus là 'expand' (mở rộng 100 mẫu)
if AUDIT is None or AUDIT.get('decision') != 'expand':
    print('Không cần round 2 (decision hiện tại: %s) — bỏ qua.'
          % (AUDIT.get('decision') if AUDIT else 'chưa có kết quả audit'))
else:
    audited = {s['sample_id'] for s in a_samples}
    rest = sorted((x for x in samples if x['sample_id'] not in audited), key=lambda x: x['test_index'])
    round2 = {
        'created': RUN_STAMP, 'notebook': NOTEBOOK, 'seed': SEED,
        'n_audit': len(rest), 'rubric': RUBRIC,
        'llm_labels_available': LLM_LABELS_AVAILABLE,
        'instructions': ['Như round 1 — điền label rồi lưu thành data/audit_filled_round2.json.',
                         'Chạy lại cell đối chiếu sau khi mở rộng schema (hoặc gộp file thủ công).'],
        'samples': [sample_view(s) for s in rest],
    }
    _blob = json.dumps(round2, ensure_ascii=False)
    assert 'llm_confidence' not in _blob and 'llm_probabilities' not in _blob
    p2 = OUTPUT_DIR / 'audit_samples_round2.json'
    p2.write_text(json.dumps(round2, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Đã xuất %s: %d mẫu blind còn lại (đủ 100).' % (p2, len(rest)))
